In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz
dbutils.widgets.removeAll()
# Parámetros del Job
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# 1. Parámetros del Job
env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# 2. Lectura de Bronze
df_bronze = spark.table(f"{catalog}.bronze.transacciones_bronze").filter(col("process_date") == process_date)

# 3. Transformaciones y Limpieza
df_silver = df_bronze.select(
    col("id_cliente").cast("bigint"),
    # Generamos el ID de tarjeta
    abs(hash(col("id_cliente"), col("indice_tarjeta"))).alias("id_tarjeta"),
    # Reconstrucción del Timestamp
    to_timestamp(
        concat(col("ano"), lit("-"), col("mes"), lit("-"), col("dia"), lit(" "), 
        substring(col("hora"), -8, 8)), 
        "yyyy-M-d HH:mm:ss"
    ).alias("fecha_completa"),
    # Limpiamos el monto (una sola vez)
    regexp_replace(col("monto"), r"[\$,]", "").cast("double").alias("monto_clean"),
    col("codigo_mcc").cast("string").alias("categoria_mcc"),
    # Metadatos de tiempo
    date_format(to_date(concat(col("ano"), lit("-"), col("mes"), lit("-"), col("dia")), "yyyy-M-d"), "EEEE").alias("dia_semana"),
    when(hour(to_timestamp(col("hora"), "HH:mm")).between(22, 23) | hour(to_timestamp(col("hora"), "HH:mm")).between(0, 4), True).otherwise(False).alias("hora_pico"),
    when(col("es_fraude") == "SÍ", "1").otherwise("0").alias("es_fraude"),
    current_timestamp().alias("ingestion_timestamp"),
    lit(process_date).alias("process_date")
).select(
    # Re-seleccionamos para renombrar el monto limpio y calcular el relativo
    "*",
    (col("monto_clean") / 100).alias("monto_relativo")
).withColumnRenamed("monto_clean", "monto")

# 4. 🔥 PASO CLAVE: DEDUPLICACIÓN 🔥
# Esto evita el error del Job eliminando filas que tengan el mismo cliente y misma hora exacta
df_silver_unique = df_silver.dropDuplicates(["id_cliente", "fecha_completa"])

# 5. MERGE en la capa Silver
target_path = f"{catalog}.silver.transacciones_silver"
dt = DeltaTable.forName(spark, target_path)

print(f"🚀 Iniciando MERGE en {target_path}...")

dt.alias("t").merge(
    df_silver_unique.alias("s"), 
    "t.id_cliente = s.id_cliente AND t.fecha_completa = s.fecha_completa"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print(f"✅ Proceso completado exitosamente para la fecha {process_date}.")

In [0]:
# 1. Consultamos la tabla física (usando target_path que es como la llamaste arriba)
df_final_silver = spark.table(target_path)

# 2. Conteo total
total_registros_silver = df_final_silver.count()

# 3. Conteo de la fecha de proceso actual
registros_hoy = df_final_silver.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" 📊 REPORTE DE TABLA FÍSICA: {target_path.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Ruta de la Tabla':<35} : {target_path}")
print(f"{'Total registros acumulados':<35} : {total_registros_silver:,}")
print(f"{'Registros procesados hoy':<35} : {registros_hoy:,}")
print(f"{'Última sincronización (Perú)':<35} : {ts_peru}")
print("═" * 70)

# 4. Vista previa
print(f"\n🔍 VISTA PREVIA DE LA TABLA SILVER (Top 3 recientes):")
display(
    df_final_silver
    .orderBy(col("ingestion_timestamp").desc())
    .limit(3)
)